In [1]:
# Phase 5: Cross-Encoder Semantic Matching

from pathlib import Path
import pandas as pd
import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "requirement_evidence.csv"
OUTPUT_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Remove rows without usable evidence text
df = df.dropna(subset=["requirement_text", "evidence_text"]).copy()
df["requirement_text"] = df["requirement_text"].astype(str).str.strip()
df["evidence_text"] = df["evidence_text"].astype(str).str.strip()
df = df[
    (df["requirement_text"] != "") &
    (df["evidence_text"] != "")
].copy()

print("Usable rows:", len(df))
print("\nLabels:")
print(df["final_evidence_label"].value_counts())
print("\nDevice:", "CUDA" if torch.cuda.is_available() else "CPU")

c:\Users\Siddh\Desktop\BidGuard-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Usable rows: 108562

Labels:
final_evidence_label
SUPPORTS              74864
DOES_NOT_SUPPORT      17829
PARTIALLY_SUPPORTS    15869
Name: count, dtype: int64

Device: CPU


In [2]:
from sklearn.model_selection import train_test_split

# Stratified sample: preserve label proportions
SAMPLE_SIZE = min(5000, len(df))

sample_df, _ = train_test_split(
    df,
    train_size=SAMPLE_SIZE,
    stratify=df["final_evidence_label"],
    random_state=42
)

sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))
print("\nSample label distribution:")
print(sample_df["final_evidence_label"].value_counts())

Sample size: 5000

Sample label distribution:
final_evidence_label
SUPPORTS              3448
DOES_NOT_SUPPORT       821
PARTIALLY_SUPPORTS     731
Name: count, dtype: int64


In [3]:
# Load Cross-Encoder model
from sentence_transformers import CrossEncoder

CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

cross_encoder = CrossEncoder(
    CROSS_ENCODER_NAME,
    device="cpu"
)

print("Cross-Encoder loaded:", CROSS_ENCODER_NAME)

c:\Users\Siddh\Desktop\BidGuard-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Siddh\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Siddh\Desktop\BidGuard-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py

Cross-Encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [4]:
# Generate Cross-Encoder relevance scores

pairs = list(zip(
    sample_df["requirement_text"].tolist(),
    sample_df["evidence_text"].tolist()
))

cross_scores = cross_encoder.predict(
    pairs,
    batch_size=16,
    show_progress_bar=True
)

sample_df["cross_encoder_score"] = np.asarray(cross_scores).reshape(-1)

print("Scored pairs:", len(sample_df))
print(sample_df[
    ["requirement_text", "evidence_text",
     "final_evidence_label", "cross_encoder_score"]
].head(10).to_string(index=False))

Batches: 100%|██████████| 313/313 [00:46<00:00,  6.80it/s]

Scored pairs: 5000
                                                                        requirement_text                                                                                                                                                                                                                                                               evidence_text final_evidence_label  cross_encoder_score
       Where applicable, the bidder shall provide valid EPFO/ESIC registration evidence.                                                                                                                                DOCUMENT TITLE: Supporting Document — NexoraSystems Pvt Ltd 0651\nSynthetic research document. No real government credential is represented.     DOES_NOT_SUPPORT           -11.006556
                              A valid tax identity document is required for eligibility.                                                                  CERTIFICATE / ELIGIBILITY DET

In [5]:
# Inspect score distribution
print("Score statistics:")
print(sample_df["cross_encoder_score"].describe())

print("\nMean score by ground-truth label:")
print(
    sample_df.groupby("final_evidence_label")["cross_encoder_score"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)

print("\nHighest relevance scores:")
display(
    sample_df.nlargest(5, "cross_encoder_score")[
        ["requirement_text", "evidence_text",
         "final_evidence_label", "cross_encoder_score"]
    ]
)

print("\nLowest relevance scores:")
display(
    sample_df.nsmallest(5, "cross_encoder_score")[
        ["requirement_text", "evidence_text",
         "final_evidence_label", "cross_encoder_score"]
    ]
)

Score statistics:
count    5000.000000
mean       -9.905487
std         2.432874
min       -11.418777
25%       -11.244791
50%       -10.958257
75%        -9.822367
max         3.630376
Name: cross_encoder_score, dtype: float64

Mean score by ground-truth label:
                      count   mean  median    std
final_evidence_label                             
DOES_NOT_SUPPORT        821 -9.842 -10.832  2.440
PARTIALLY_SUPPORTS      731 -9.795 -10.919  2.576
SUPPORTS               3448 -9.944 -11.003  2.399

Highest relevance scores:


,requirement_text,evidence_text,final_evidence_label,cross_encoder_score
1328,OEM authorization covering the offered product...,CERTIFICATE / ELIGIBILITY DETAILS\nauthorizati...,SUPPORTS,3.630376
4699,OEM authorization covering the offered product...,CERTIFICATE / ELIGIBILITY DETAILS\nauthorizati...,SUPPORTS,3.123986
307,OEM authorization covering the offered product...,CERTIFICATE / ELIGIBILITY DETAILS\nauthorizati...,SUPPORTS,2.960341
1294,OEM authorization covering the offered product...,CERTIFICATE / ELIGIBILITY DETAILS\nauthorizati...,SUPPORTS,2.960328
3315,OEM authorization covering the offered product...,CERTIFICATE / ELIGIBILITY DETAILS\nauthorizati...,SUPPORTS,2.948131



Lowest relevance scores:


,requirement_text,evidence_text,final_evidence_label,cross_encoder_score
1788,"Where MSME status is claimed, valid Udyam evid...",BUSINESS NARRATIVE\nThe supplier describes cap...,SUPPORTS,-11.418777
3980,"Where MSME status is claimed, valid Udyam evid...",BUSINESS NARRATIVE\nThe supplier describes cap...,DOES_NOT_SUPPORT,-11.415285
2552,"Where MSME status is claimed, valid Udyam evid...",CERTIFICATE / ELIGIBILITY DETAILS\nfinancial_y...,SUPPORTS,-11.410833
425,"Where MSME status is claimed, valid Udyam evid...",BUSINESS NARRATIVE\nThe supplier describes cap...,SUPPORTS,-11.409030
4046,"Where MSME status is claimed, valid Udyam evid...",BUSINESS NARRATIVE\nThe supplier describes cap...,DOES_NOT_SUPPORT,-11.409030


In [6]:
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr

label_order = {
    "DOES_NOT_SUPPORT": 0,
    "PARTIALLY_SUPPORTS": 1,
    "SUPPORTS": 2
}

sample_df["label_rank"] = sample_df["final_evidence_label"].map(label_order)

correlation = spearmanr(
    sample_df["cross_encoder_score"],
    sample_df["label_rank"]
)

print("Spearman correlation:", round(correlation.statistic, 4))
print("P-value:", correlation.pvalue)

print("\nScore ranges by label:")
print(
    sample_df.groupby("final_evidence_label")["cross_encoder_score"]
    .agg(["min", "median", "max"])
    .round(3)
)

Spearman correlation: -0.0588
P-value: 3.190961989561038e-05

Score ranges by label:
                         min  median    max
final_evidence_label                       
DOES_NOT_SUPPORT     -11.415 -10.832  2.735
PARTIALLY_SUPPORTS   -11.396 -10.919  1.951
SUPPORTS             -11.419 -11.003  3.630


In [7]:
# Create binary relevance evaluation dataset

binary_df = sample_df[
    sample_df["final_evidence_label"].isin(
        ["SUPPORTS", "DOES_NOT_SUPPORT"]
    )
].copy()

binary_df["binary_label"] = (
    binary_df["final_evidence_label"] == "SUPPORTS"
).astype(int)

print("Evaluation rows:", len(binary_df))
print("\nBinary label counts:")
print(binary_df["binary_label"].value_counts().rename({
    1: "Relevant (SUPPORTS)",
    0: "Not relevant (DOES_NOT_SUPPORT)"
}))

Evaluation rows: 4269

Binary label counts:
binary_label
Relevant (SUPPORTS)                3448
Not relevant (DOES_NOT_SUPPORT)     821
Name: count, dtype: int64


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

y_true = binary_df["binary_label"].to_numpy()
y_score = binary_df["cross_encoder_score"].to_numpy()

# Initial baseline: score > 0 means relevant
y_pred = (y_score > 0).astype(int)

print("Initial Cross-Encoder Baseline")
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("PR-AUC / Average Precision:", round(average_precision_score(y_true, y_score), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=["DOES_NOT_SUPPORT", "SUPPORTS"],
    zero_division=0
))

Initial Cross-Encoder Baseline
Accuracy: 0.2026
Precision: 0.7973
Recall: 0.0171
F1: 0.0335
PR-AUC / Average Precision: 0.7953

Confusion Matrix:
[[ 806   15]
 [3389   59]]

Classification Report:
                  precision    recall  f1-score   support

DOES_NOT_SUPPORT       0.19      0.98      0.32       821
        SUPPORTS       0.80      0.02      0.03      3448

        accuracy                           0.20      4269
       macro avg       0.49      0.50      0.18      4269
    weighted avg       0.68      0.20      0.09      4269



In [9]:
from sklearn.model_selection import train_test_split

# Stratified holdout: threshold tuning and final testing stay separate
train_df, test_df = train_test_split(
    binary_df,
    test_size=0.25,
    random_state=42,
    stratify=binary_df["binary_label"]
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining labels:")
print(train_df["binary_label"].value_counts())

print("\nTesting labels:")
print(test_df["binary_label"].value_counts())

Training rows: 3201
Testing rows: 1068

Training labels:
binary_label
1    2585
0     616
Name: count, dtype: int64

Testing labels:
binary_label
1    863
0    205
Name: count, dtype: int64


In [10]:
from sklearn.metrics import precision_recall_curve

# Tune threshold using training data only
y_train = train_df["binary_label"].to_numpy()
scores_train = train_df["cross_encoder_score"].to_numpy()

precision, recall, thresholds = precision_recall_curve(
    y_train, scores_train
)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
best_threshold = float(thresholds[best_idx])

print("Best training threshold:", round(best_threshold, 4))
print("Training precision:", round(precision[best_idx], 4))
print("Training recall:", round(recall[best_idx], 4))
print("Training F1:", round(f1_values[best_idx], 4))


Best training threshold: -11.4188
Training precision: 0.8076
Training recall: 1.0
Training F1: 0.8935


In [11]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, average_precision_score,
    confusion_matrix, classification_report
)

y_test = test_df["binary_label"].to_numpy()
scores_test = test_df["cross_encoder_score"].to_numpy()
pred_test = (scores_test >= best_threshold).astype(int)

print("TEST RESULTS — Tuned Threshold")
print("Threshold:", round(best_threshold, 4))
print("Accuracy:", round(accuracy_score(y_test, pred_test), 4))
print("Precision:", round(precision_score(y_test, pred_test, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred_test, zero_division=0), 4))
print("F1:", round(f1_score(y_test, pred_test, zero_division=0), 4))
print("PR-AUC:", round(average_precision_score(y_test, scores_test), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_test))

print("\nClassification Report:")
print(classification_report(
    y_test, pred_test,
    labels=[0, 1],
    target_names=["DOES_NOT_SUPPORT", "SUPPORTS"],
    zero_division=0
))

TEST RESULTS — Tuned Threshold
Threshold: -11.4188
Accuracy: 0.8081
Precision: 0.8081
Recall: 1.0
F1: 0.8938
PR-AUC: 0.7975

Confusion Matrix:
[[  0 205]
 [  0 863]]

Classification Report:
                  precision    recall  f1-score   support

DOES_NOT_SUPPORT       0.00      0.00      0.00       205
        SUPPORTS       0.81      1.00      0.89       863

        accuracy                           0.81      1068
       macro avg       0.40      0.50      0.45      1068
    weighted avg       0.65      0.81      0.72      1068



In [12]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import balanced_accuracy_score

# Tune threshold on training set using balanced accuracy
y_train = train_df["binary_label"].to_numpy()
scores_train = train_df["cross_encoder_score"].to_numpy()

candidate_thresholds = np.unique(
    np.quantile(scores_train, np.linspace(0.01, 0.99, 200))
)

results = []

for threshold in candidate_thresholds:
    pred = (scores_train >= threshold).astype(int)

    results.append({
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_train, pred),
        "precision": precision_score(y_train, pred, zero_division=0),
        "recall": recall_score(y_train, pred, zero_division=0),
        "f1": f1_score(y_train, pred, zero_division=0)
    })

threshold_results = pd.DataFrame(results)

# Prefer the threshold with best balanced accuracy
best_row = threshold_results.loc[
    threshold_results["balanced_accuracy"].idxmax()
]
balanced_threshold = float(best_row["threshold"])

print("Balanced threshold:", round(balanced_threshold, 4))
print(best_row.to_string())

Balanced threshold: -7.3553
threshold           -7.355254
balanced_accuracy    0.505886
precision            0.823691
recall               0.115667
f1                   0.202849


In [13]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    average_precision_score
)

y_test = test_df["binary_label"].to_numpy()
scores_test = test_df["cross_encoder_score"].to_numpy()

pred_balanced = (scores_test >= balanced_threshold).astype(int)

print("Balanced-threshold TEST results")
print("Balanced accuracy:",
      round(balanced_accuracy_score(y_test, pred_balanced), 4))
print("PR-AUC:",
      round(average_precision_score(y_test, scores_test), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred_balanced))

print("\nClassification report:")
print(classification_report(
    y_test,
    pred_balanced,
    labels=[0, 1],
    target_names=["DOES_NOT_SUPPORT", "SUPPORTS"],
    zero_division=0
))

Balanced-threshold TEST results
Balanced accuracy: 0.4999
PR-AUC: 0.7975

Confusion matrix:
[[180  25]
 [758 105]]

Classification report:
                  precision    recall  f1-score   support

DOES_NOT_SUPPORT       0.19      0.88      0.31       205
        SUPPORTS       0.81      0.12      0.21       863

        accuracy                           0.27      1068
       macro avg       0.50      0.50      0.26      1068
    weighted avg       0.69      0.27      0.23      1068



In [14]:
# Save Phase 5 evaluation artifacts

import json

results_dir = PROJECT_ROOT / "data" / "processed"
models_dir = PROJECT_ROOT / "models"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# Save scored evaluation sample
sample_df.to_csv(
    results_dir / "cross_encoder_scored_sample.csv",
    index=False
)

# Save evaluation summary
summary = {
    "model": CROSS_ENCODER_NAME,
    "sample_size": int(len(sample_df)),
    "binary_evaluation_size": int(len(binary_df)),
    "train_size": int(len(train_df)),
    "test_size": int(len(test_df)),
    "threshold_strategy": "balanced_accuracy",
    "selected_threshold": float(balanced_threshold),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(y_test, pred_balanced)
    ),
    "test_pr_auc": float(
        average_precision_score(y_test, scores_test)
    ),
    "test_precision_supports": float(
        precision_score(y_test, pred_balanced, zero_division=0)
    ),
    "test_recall_supports": float(
        recall_score(y_test, pred_balanced, zero_division=0)
    ),
    "test_f1_supports": float(
        f1_score(y_test, pred_balanced, zero_division=0)
    )
}

with open(
    results_dir / "cross_encoder_evaluation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print("Saved scored sample and evaluation summary.")
print(results_dir / "cross_encoder_scored_sample.csv")
print(results_dir / "cross_encoder_evaluation.json")

Saved scored sample and evaluation summary.
c:\Users\Siddh\Desktop\BidGuard-AI\data\processed\cross_encoder_scored_sample.csv
c:\Users\Siddh\Desktop\BidGuard-AI\data\processed\cross_encoder_evaluation.json
